# Real Wikinews classification: latent topics and dense representations

Companion to the lexical notebook. This notebook tests **truncated SVD**, **LDA document-topic mixtures**, and **augmented SVD components** using the same 15 licensed Wikinews excerpts and locally annotated politics/nonpolitics label.

The question is not whether these representations produce impressive accuracy on 15 articles. It is whether a researcher can inspect a latent representation, compare model families, preserve key alignment, freeze a selected classifier, and evaluate it on separately held-out annotations using TeAL.

**Important:** Vocabulary, LDA and SVD are estimated using the **full unlabeled article corpus**, so this is a *transductive finite-corpus* exercise. In a prospective setting these unsupervised estimators would instead have to be fitted on training data only. No label column enters the feature matrices.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

from text_analysis_lab import Project
from text_analysis_lab.analysis import ClassifierSearch
from text_analysis_lab.translators import (
    CountVectorizer, SVD, LDA, TfidfTransformer, FeatureMatrix,
)

def find_root():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / "tests/fixtures/wikinews_2025_small/articles.jsonl").exists():
            return root
    raise RuntimeError("Notebook must run inside the TeAL checkout.")

ROOT = find_root()
PATH = ROOT / "tests/fixtures/wikinews_2025_small/articles.jsonl"
WORK = Path(tempfile.mkdtemp(prefix="teal-wikinews-latent-"))
print("Work directory:", WORK)

In [ ]:
articles = pd.DataFrame([
    json.loads(line) for line in PATH.read_text(encoding="utf-8").splitlines() if line.strip()
]).sort_values("article_id").reset_index(drop=True)
articles["is_politics"] = articles["topic_group"].eq("politics").astype(int)
articles["n_chars"] = articles["text"].str.len()
print("Document count:", len(articles))
display(articles[["article_id", "title", "topic_group"]])
assert len(articles) == 15

## TeAL artifact registration and protected labels

Only keys in the *development* subset enter `ClassifierSearch`. The audit subset is kept separate until the representation/model search is frozen.

In [ ]:
file = WORK / "wikinews.csv"
articles[["text", "article_id", "topic_group", "is_politics", "n_chars"]].to_csv(file, index=False)
project = Project.create(WORK / "project", name="wikinews_latent_acceptance")
source = project.read_csv(
    file, text_fields="text",
    metadata_fields=["article_id", "topic_group", "is_politics", "n_chars"],
)
slices = project.split(
    source,
    labels=("development", "audit"),
    proportions=(0.7, 0.3), random_state=2025,
    stratify="is_politics",
)
development, audit = slices["development"], slices["audit"]
assert development.n_rows + audit.n_rows == 15

## Build and inspect learned latent features

A count matrix feeds **LDA** (nonnegative input); normalized TF-IDF feeds **SVD**. Both emit dense document representations. We also test a computed dense matrix that appends log document length to SVD coordinates. The length is independent of the ground-truth topic label.

These topic components are exploratory: their labels/meanings must be interpreted by looking at their highest-weighted terms, not by treating component numbers as substantive constructs.

In [ ]:
counts = project.translate(
    CountVectorizer(text_field="text", stop_words="english", max_features=160),
    source,
)["output"]
tfidf = project.translate(TfidfTransformer(norm="l2"), counts)["output"]
svd = project.translate(SVD(n_components=3, random_state=2025), tfidf)["output"]
lda = project.translate(
    LDA(n_components=3, max_iter=15, random_state=2025), counts,
)["output"]
enhanced = project.translate(
    FeatureMatrix(
        data_columns=True, metadata_columns=["n_chars"], matrix_type="dense",
    ), svd,
)["output"]
for name, representation in (("svd", svd), ("lda", lda), ("svd_plus_length", enhanced)):
    print(name, representation.get_matrix().shape)
    assert representation.get_matrix().shape[0] == 15
assert svd.get_matrix().shape[1] == lda.get_matrix().shape[1] == 3
assert enhanced.get_matrix().shape[1] == 4
display(pd.DataFrame(lda.get_matrix(), columns=lda.get_data_columns()).describe().round(3))

## Fixed-fold comparison of two model families

The same folds and development observations are used for SVD, LDA, and enhanced SVD. We use small hyperparameter grids and report failures rather than hiding them. No audit labels are consulted during selection.

In [ ]:
search = ClassifierSearch(
    representations=[svd, lda, enhanced],
    models={
        "logistic": {"C": [0.1, 1.0, 10.0]},
        "linear_svm": {"C": [0.5, 1.0]},
    },
    cv=2, scoring="f1", random_state=2025, batch_size=4,
)
display(search.plan(development, target="is_politics")[
    ["candidate", "representation_id", "model", "params", "planned_fits"]
])
search.fit(development, target="is_politics")
results = search.results().sort_values(
    ["mean_score", "std_score"], ascending=[False, True],
).reset_index(drop=True)
display(results[["candidate", "representation_id", "model", "params",
                 "mean_score", "std_score", "status", "error"]])
assert results["status"].eq("ok").any()

## Fresh development refit and untouched audit evaluation

Now that the model-selection decision has been made, the audit labels can be accessed for descriptive inspection. The small number of audit items means a single mistake changes the reported score substantially.

In [ ]:
winner = int(results.loc[results["status"] == "ok", "candidate"].iloc[0])
chosen = search.finalize(candidate=winner)
assert chosen.provenance["development_rows"] == development.n_rows
predictions = project.translate(chosen.predictor, chosen.source)["output"]

gold = audit.query(
    key_columns=True, data_columns=False,
    metadata_columns=["is_politics", "topic_group", "article_id"],
    metadata_mode="full", form="table",
)
scores = predictions.query(
    key_columns=True, data_columns=["prediction"], form="table",
)
audit_results = gold.merge(
    scores, on=list(audit.primary_key), validate="one_to_one",
)
display(audit_results[["article_id", "topic_group", "is_politics", "prediction"]])
assert len(audit_results) == audit.n_rows
print(classification_report(
    audit_results["is_politics"], audit_results["prediction"], zero_division=0,
))

## Frozen model survives reopening

The fitted estimator is saved once, with its training provenance. Verify that it yields the same predictions after reopening the TeAL Project.

In [ ]:
op_id = project.catalog.get_operation(
    predictions.descriptor["operation_id"]
)["operator_id"]
prediction_values = scores["prediction"].to_numpy(copy=True)
source_id = chosen.source.artifact_id
project.close()
reopened = Project.open(WORK / "project")
restored = reopened.get_operator(op_id)
assert restored.training_provenance == chosen.provenance
again = reopened.translate(restored, reopened.get_artifact(source_id))["output"]
repeat = again.query(data_columns=["prediction"], key_columns=False, form="table")
np.testing.assert_array_equal(prediction_values, repeat["prediction"].to_numpy())
print("Frozen predictor replayed", len(repeat), "article predictions.")
reopened.close()

## Qualitative acceptance questions

Which model and representation won this *development-only* comparison? Are the LDA mixtures interpretable for this corpus? Do reduced SVD dimensions plausibly separate article types? Where did the final classifier fail on audit cases, and what alternative representations should be tried on **new** development data rather than repeatedly tuning against the audit?

**Do not present these results as evidence that one representation family is generally superior.** The purpose is to catch real-world interface, feature, persistence, and leakage failures.